# Calculadora de Ancestralidade Genômica

## 1. Configuração e validação dos dados

Este notebook é portátil: usa `E:\ATV1\Data` localmente e o Google Drive no Colab. Nesta etapa não processamos os arquivos brutos de `referencia/`, porque a disciplina já forneceu a referência reduzida em `processed/`.

In [1]:
from pathlib import Path
import os

LOCAL_BASE = Path(r"E:\ATV1\Data")
COLAB_BASE = Path("/content/drive/MyDrive/Projeto_Ancestralidade")

if LOCAL_BASE.exists():
    BASE = LOCAL_BASE
    AMBIENTE = "local"
else:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise FileNotFoundError(
            f"Não encontrei os dados locais em {LOCAL_BASE}. Defina ANCESTRY_DATA_DIR ou execute no Colab."
        ) from exc
    drive.mount("/content/drive")
    BASE = Path(os.environ.get("ANCESTRY_DATA_DIR", COLAB_BASE))
    AMBIENTE = "Colab"

INDIVIDUO = BASE / "individuo"
PROCESSED = BASE / "processed"
GENOTIPO = INDIVIDUO / "genotipo_microarray.csv"
PGEN = PROCESSED / "1000G_GSA_SNP.pgen"
PVAR = PROCESSED / "1000G_GSA_SNP.pvar.zst"
PSAM = PROCESSED / "1000G_GSA_SNP.psam"
MAPA = PROCESSED / "harmonization_map.csv"
METADATA = PROCESSED / "metadata_populations.csv"

print(f"Ambiente: {AMBIENTE}")
print(f"Base de dados: {BASE}")

Ambiente: local
Base de dados: E:\ATV1\Data


In [2]:
arquivos_necessarios = [GENOTIPO, PGEN, PVAR, PSAM, MAPA, METADATA]
faltantes = [arquivo for arquivo in arquivos_necessarios if not arquivo.exists()]

if faltantes:
    raise FileNotFoundError("Arquivos ausentes:\n" + "\n".join(map(str, faltantes)))

for arquivo in arquivos_necessarios:
    print(f"✓ {arquivo.relative_to(BASE)} ({arquivo.stat().st_size / 1e6:.1f} MB)")

✓ individuo\genotipo_microarray.csv (28.9 MB)
✓ processed\1000G_GSA_SNP.pgen (269.4 MB)
✓ processed\1000G_GSA_SNP.pvar.zst (113.5 MB)
✓ processed\1000G_GSA_SNP.psam (0.1 MB)
✓ processed\harmonization_map.csv (32.3 MB)
✓ processed\metadata_populations.csv (0.1 MB)


In [3]:
import pandas as pd

metadata = pd.read_csv(METADATA)
mapa = pd.read_csv(MAPA)
genotipo_exemplo = pd.read_csv(GENOTIPO, nrows=5)

assert metadata.shape[0] == 3202, f"Esperados 3202 indivíduos; obtidos {metadata.shape[0]}"
assert mapa.shape[0] == 572011, f"Esperados 572011 SNPs; obtidos {mapa.shape[0]}"
assert mapa["INDEX"].is_unique, "O mapa de harmonização tem índices duplicados."
assert set(metadata["SuperPop"]) == {"AFR", "AMR", "EAS", "EUR", "SAS"}

print("Validação estrutural concluída.")
print(f"Populações: {metadata['Population'].nunique()}")
print(f"Superpopulações: {', '.join(sorted(metadata['SuperPop'].unique()))}")
display(genotipo_exemplo)
display(metadata.head())
display(mapa.head())

Validação estrutural concluída.
Populações: 26
Superpopulações: AFR, AMR, EAS, EUR, SAS


,Index,Name,Address,Chr,Position,35.Genotipo
0,1,1:103380393,9663149,1,102914837,GG
1,2,1:109439680,26641362,1,108897058,AA
2,3,1:110198788,5682188,1,109656166,TT
3,4,1:110201112,21701591,1,109658490,CC
4,5,1:110201667,79803181,1,109659045,CC


,Sample,Population,SuperPop,SEX
0,HG00096,GBR,EUR,Male
1,HG00097,GBR,EUR,Female
2,HG00099,GBR,EUR,Female
3,HG00100,GBR,EUR,Female
4,HG00101,GBR,EUR,Male


,INDEX,Name,CHR,POS,ID,REF,ALT,NEEDS_COMPLEMENT
0,0,GSA-rs116587930,1,792461,1:792461:G:A,G,A,True
1,1,rs3131972,1,817341,1:817341:A:G,A,G,False
2,2,GSA-rs114525117,1,823656,1:823656:G:A,G,A,True
3,3,rs12127425,1,858952,1:858952:G:A,G,A,True
4,4,GSA-rs79373928,1,866156,1:866156:T:G,T,G,False


### Próximo passo

Ler os genótipos da referência PLINK 2 sem esgotar a memória, aplicar os índices de `harmonization_map.csv` e construir `X_ref_final` com dimensão esperada `(3202, 572011)`.